# Week 6: Sorting — Insertion, Merge & Quick Sort — PHASE 3 "Choosing Between O(n), O(log n), O(1)"

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Understand **why sorting matters** and where it is used
2. Implement **Insertion Sort** and trace its step-by-step behavior
3. Understand the **Merge Sort** algorithm: divide, sort halves, merge
4. Understand the **Quick Sort** algorithm: pick pivot, partition, recurse
5. Compare sorting algorithms using **Big-O notation**
6. Use Python's built-in `sorted()` and `.sort()` (Timsort)
7. Benchmark Insertion Sort vs Python's built-in sort and interpret results

## 🎯 Core Mastery Connection

Different sorts win in different situations — know the trade-offs, measure them. Insertion Sort is O(n^2) but shines on nearly-sorted data; Merge Sort guarantees O(n log n) but uses extra memory; Quick Sort is fast in practice but has a dangerous worst case. You will benchmark all three and see that "fastest" depends on the data. This is the heart of empirical complexity reasoning: theory tells you the Big-O, but benchmarks reveal the real-world winner.

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

import math
import random
import time

---
## Part 1: Why Does Sorting Matter?

Sorting is one of the most fundamental operations in computer science. Once data is sorted, many other operations become much faster.

### Where Sorting Is Used

| Application | Why Sorting Helps |
|---|---|
| Binary search | Requires sorted data to work |
| Finding duplicates | Duplicates end up next to each other |
| Finding the median | Middle element of sorted data |
| Database queries | ORDER BY, GROUP BY rely on sorting |
| Leaderboards | Ranking players by score |
| Autocomplete | Alphabetically sorted suggestions |

### The Sorting Problem

**Input:** A list of items in arbitrary order  
**Output:** The same items rearranged in non-decreasing order

```
Input:  [38, 27, 43, 3, 9, 82, 10]
Output: [3, 9, 10, 27, 38, 43, 82]
```

---
## Part 2: Insertion Sort — The Card-Sorting Analogy

### The Analogy

Imagine you're holding playing cards:
1. Pick up cards one at a time from the table
2. Each new card: slide it into the correct position among the cards already in your hand
3. Your hand is always sorted!

### How It Works

1. Start with the second element (index 1)
2. Compare it with elements to its left
3. Shift larger elements one position to the right
4. Insert the current element in its correct position
5. Repeat for all remaining elements

**Figure 2.1** — Insertion sort implementation

In [ ]:
def insertion_sort(lst):
    """Sort a list in-place using insertion sort."""
    for i in range(1, len(lst)):
        key = lst[i]           # The card we're inserting
        j = i - 1
        
        # Shift elements that are greater than key to the right
        while j >= 0 and lst[j] > key:
            lst[j + 1] = lst[j]
            j -= 1
        
        lst[j + 1] = key       # Place the card in correct position
    
    return lst

# Test it
data = [38, 27, 43, 3, 9, 82, 10]
print(f"Before: {data}")
insertion_sort(data)
print(f"After:  {data}")

**Figure 2.2** — Tracing insertion sort step by step

In [ ]:
def insertion_sort_traced(lst):
    """Insertion sort with step-by-step visualization."""
    lst = lst.copy()  # Don't modify original
    print(f"Start:  {lst}")
    print("=" * 50)
    
    for i in range(1, len(lst)):
        key = lst[i]
        j = i - 1
        
        print(f"\nStep {i}: Insert {key}")
        print(f"  Sorted part: {lst[:i]}  |  Unsorted: {lst[i:]}")
        
        shifts = 0
        while j >= 0 and lst[j] > key:
            lst[j + 1] = lst[j]
            j -= 1
            shifts += 1
        
        lst[j + 1] = key
        
        if shifts > 0:
            print(f"  Shifted {shifts} element(s), placed {key} at index {j+1}")
        else:
            print(f"  {key} is already in correct position")
        print(f"  Result: {lst}")
    
    print("\n" + "=" * 50)
    print(f"Final:  {lst}")
    return lst

insertion_sort_traced([5, 2, 8, 1, 9, 3])

### Insertion Sort Complexity

| Case | Comparisons | Big-O | When? |
|---|---|---|---|
| Best case | n - 1 | O(n) | Already sorted |
| Worst case | n(n-1)/2 | O(n²) | Reverse sorted |
| Average case | ~n²/4 | O(n²) | Random order |

**Good for:** Small lists, nearly-sorted data  
**Bad for:** Large, randomly ordered lists

**Figure 2.3** — Counting comparisons in insertion sort

In [ ]:
def insertion_sort_count(lst):
    """Insertion sort that counts comparisons and shifts."""
    lst = lst.copy()
    comparisons = 0
    shifts = 0
    
    for i in range(1, len(lst)):
        key = lst[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if lst[j] > key:
                lst[j + 1] = lst[j]
                shifts += 1
                j -= 1
            else:
                break
        lst[j + 1] = key
    
    return lst, comparisons, shifts

import random

# Compare best, worst, and average cases
n = 20
best_case = list(range(n))                # Already sorted
worst_case = list(range(n, 0, -1))        # Reverse sorted
avg_case = random.sample(range(n), n)     # Random

for name, data in [("Best (sorted)", best_case), ("Worst (reversed)", worst_case), ("Average (random)", avg_case)]:
    _, comps, shifts = insertion_sort_count(data)
    print(f"{name:20s}: {comps:4d} comparisons, {shifts:4d} shifts")

---
## Part 3: Merge Sort — Divide, Sort, Merge

Merge Sort uses a **divide and conquer** strategy:

1. **Divide:** Split the list into two halves
2. **Conquer:** Recursively sort each half
3. **Merge:** Combine the two sorted halves into one sorted list

### The Analogy

Imagine sorting a deck of cards:
1. Split the deck in half
2. Sort each half separately
3. Merge by comparing the top card of each pile and picking the smaller one

```
       [38, 27, 43, 3, 9, 82, 10]
              /            \
      [38, 27, 43, 3]   [9, 82, 10]
        /       \          /      \
    [38, 27]  [43, 3]  [9, 82]  [10]
     / \       / \      / \
   [38][27] [43][3]  [9][82]   [10]
     \ /       \ /      \ /
    [27, 38]  [3, 43]  [9, 82]  [10]
        \       /          \      /
      [3, 27, 38, 43]   [9, 10, 82]
              \            /
       [3, 9, 10, 27, 38, 43, 82]
```

**Figure 3.1** — Merge sort implementation

In [ ]:
def merge_sort(lst):
    """Sort a list using merge sort. Returns a new sorted list."""
    # Base case: list of 0 or 1 elements is already sorted
    if len(lst) <= 1:
        return lst
    
    # Divide: split into two halves
    mid = len(lst) // 2
    left = merge_sort(lst[:mid])    # Sort left half
    right = merge_sort(lst[mid:])   # Sort right half
    
    # Merge: combine two sorted halves
    return merge(left, right)

def merge(left, right):
    """Merge two sorted lists into one sorted list."""
    result = []
    i = j = 0
    
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    
    # Add remaining elements
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# Test it
data = [38, 27, 43, 3, 9, 82, 10]
print(f"Before: {data}")
sorted_data = merge_sort(data)
print(f"After:  {sorted_data}")

**Figure 3.2** — Tracing the merge sort process

In [ ]:
def merge_sort_traced(lst, depth=0):
    """Merge sort with visual tracing."""
    indent = "  " * depth
    
    if len(lst) <= 1:
        print(f"{indent}Base: {lst}")
        return lst
    
    mid = len(lst) // 2
    print(f"{indent}Split: {lst} -> {lst[:mid]} + {lst[mid:]}")
    
    left = merge_sort_traced(lst[:mid], depth + 1)
    right = merge_sort_traced(lst[mid:], depth + 1)
    
    merged = merge(left, right)
    print(f"{indent}Merge: {left} + {right} -> {merged}")
    return merged

print("Tracing merge_sort([5, 2, 8, 1, 4]):")
print("=" * 55)
result = merge_sort_traced([5, 2, 8, 1, 4])
print("=" * 55)
print(f"Result: {result}")

### Merge Sort Complexity

| Case | Big-O | Notes |
|---|---|---|
| Best case | O(n log n) | Always divides and merges |
| Worst case | O(n log n) | Same — always consistent! |
| Average case | O(n log n) | Guaranteed performance |
| Space | O(n) | Needs extra space for merging |

**Advantage:** Guaranteed O(n log n) in all cases  
**Disadvantage:** Uses extra memory (not in-place)

---
## Part 4: Quick Sort — Pick a Pivot, Partition

Quick Sort also uses divide and conquer, but differently:

1. **Pick a pivot** element
2. **Partition:** rearrange so elements < pivot are on the left, elements > pivot are on the right
3. **Recurse:** sort the left and right partitions

### The Analogy

Imagine organizing students by height:
1. Pick one student as the "pivot"
2. Everyone shorter goes to the left, everyone taller goes to the right
3. Repeat the process within each group

**Figure 4.1** — Quick sort implementation (simple version)

In [ ]:
def quick_sort(lst):
    """Sort a list using quick sort. Returns a new sorted list."""
    # Base case
    if len(lst) <= 1:
        return lst
    
    # Pick pivot (we'll use the last element for simplicity)
    pivot = lst[-1]
    
    # Partition into three groups
    left = [x for x in lst[:-1] if x <= pivot]    # Elements <= pivot
    right = [x for x in lst[:-1] if x > pivot]    # Elements > pivot
    
    # Recurse and combine
    return quick_sort(left) + [pivot] + quick_sort(right)

# Test it
data = [38, 27, 43, 3, 9, 82, 10]
print(f"Before: {data}")
sorted_data = quick_sort(data)
print(f"After:  {sorted_data}")

**Figure 4.2** — Tracing quick sort partitioning

In [ ]:
def quick_sort_traced(lst, depth=0):
    """Quick sort with visual tracing."""
    indent = "  " * depth
    
    if len(lst) <= 1:
        print(f"{indent}Base: {lst}")
        return lst
    
    pivot = lst[-1]
    left = [x for x in lst[:-1] if x <= pivot]
    right = [x for x in lst[:-1] if x > pivot]
    
    print(f"{indent}Pivot={pivot}: {left} | [{pivot}] | {right}")
    
    sorted_left = quick_sort_traced(left, depth + 1)
    sorted_right = quick_sort_traced(right, depth + 1)
    
    result = sorted_left + [pivot] + sorted_right
    print(f"{indent}Combined: {result}")
    return result

print("Tracing quick_sort([7, 2, 1, 6, 8, 5, 3, 4]):")
print("=" * 55)
result = quick_sort_traced([7, 2, 1, 6, 8, 5, 3, 4])
print("=" * 55)
print(f"Result: {result}")

### Quick Sort Complexity

| Case | Big-O | When? |
|---|---|---|
| Best case | O(n log n) | Pivot splits evenly |
| Worst case | O(n²) | Already sorted + bad pivot choice |
| Average case | O(n log n) | Random data |
| Space | O(log n) | Recursive call stack |

**Advantage:** Very fast in practice, low memory usage  
**Disadvantage:** Worst case O(n²) with bad pivot choices

---
## Part 5: Big-O Comparison Table

| Algorithm | Best | Average | Worst | Space | Stable? |
|---|---|---|---|---|---|
| **Insertion Sort** | O(n) | O(n²) | O(n²) | O(1) | Yes |
| **Merge Sort** | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| **Quick Sort** | O(n log n) | O(n log n) | O(n²) | O(log n) | No |
| **Python sorted()** | O(n) | O(n log n) | O(n log n) | O(n) | Yes |

**Stable** means equal elements keep their original relative order.

**Key insight:** For small lists (n < 50), insertion sort can actually be faster due to low overhead!

**Figure 5.1** — Visualizing growth rates of O(n²) vs O(n log n)

In [ ]:
import matplotlib.pyplot as plt
import math

ns = list(range(1, 501))
n_squared = [n * n for n in ns]
n_log_n = [n * math.log2(n) if n > 0 else 0 for n in ns]
n_linear = ns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: All three growth rates
ax1.plot(ns, n_squared, 'r-', label='O(n\u00b2) - Insertion Sort', linewidth=2)
ax1.plot(ns, n_log_n, 'g-', label='O(n log n) - Merge/Quick Sort', linewidth=2)
ax1.plot(ns, n_linear, 'b-', label='O(n) - Best case Insertion', linewidth=2)
ax1.set_xlabel('Input Size (n)', fontsize=12)
ax1.set_ylabel('Operations', fontsize=12)
ax1.set_title('Sorting Algorithm Growth Rates', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Zoom in to small n (where insertion sort wins)
small_ns = list(range(1, 51))
ax2.plot(small_ns, [n*n for n in small_ns], 'r-', label='O(n\u00b2)', linewidth=2)
ax2.plot(small_ns, [n * math.log2(n) if n > 0 else 0 for n in small_ns], 'g-', label='O(n log n)', linewidth=2)
ax2.set_xlabel('Input Size (n)', fontsize=12)
ax2.set_ylabel('Operations', fontsize=12)
ax2.set_title('Zoomed In: Small n (n < 50)', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("For small n, the difference is small.")
print("For large n, O(n\u00b2) becomes dramatically slower than O(n log n).")

---
## Part 6: Python's Built-In Sorting — Timsort

Python's `sorted()` and `.sort()` use **Timsort**, a hybrid algorithm that combines:
- **Insertion sort** for small runs of data
- **Merge sort** for combining runs

It's highly optimized and adapts to patterns in the data.

**Figure 6.1** — Using sorted() and .sort()

In [ ]:
data = [64, 25, 12, 22, 11]

# sorted() returns a NEW sorted list (original unchanged)
new_list = sorted(data)
print(f"Original: {data}")
print(f"sorted(): {new_list}")
print(f"Original unchanged: {data}")

print()

# .sort() modifies the list IN PLACE (no new list)
data2 = [64, 25, 12, 22, 11]
data2.sort()
print(f".sort():  {data2}")
print()

# Sorting with key function
words = ["banana", "apple", "cherry", "date"]
print(f"Alphabetical:  {sorted(words)}")
print(f"By length:     {sorted(words, key=len)}")
print(f"Reverse:       {sorted(words, reverse=True)}")

**Figure 6.2** — Sorting with custom keys

In [ ]:
# Sorting dictionaries by a specific field
students = [
    {"name": "Elif", "grade": 85},
    {"name": "Ahmet", "grade": 92},
    {"name": "Zeynep", "grade": 78},
    {"name": "Can", "grade": 95},
    {"name": "Beren", "grade": 88},
]

# Sort by grade
by_grade = sorted(students, key=lambda s: s["grade"])
print("By grade (ascending):")
for s in by_grade:
    print(f"  {s['name']:>8}: {s['grade']}")

print()

# Sort by grade descending
by_grade_desc = sorted(students, key=lambda s: s["grade"], reverse=True)
print("By grade (descending):")
for s in by_grade_desc:
    print(f"  {s['name']:>8}: {s['grade']}")

**Figure 6.3** — Sorting tuples and multiple criteria

In [ ]:
# Python sorts tuples element by element
scores = [("Elif", 85), ("Ahmet", 92), ("Zeynep", 85), ("Can", 92)]

# Sort by score first, then by name (for ties)
by_score_name = sorted(scores, key=lambda x: (x[1], x[0]))
print("By score, then name:")
for name, score in by_score_name:
    print(f"  {name:>8}: {score}")

print()

# Sort by score descending, name ascending
by_score_desc = sorted(scores, key=lambda x: (-x[1], x[0]))
print("By score (desc), then name (asc):")
for name, score in by_score_desc:
    print(f"  {name:>8}: {score}")

---
## Part 7: Stability in Sorting

A **stable** sort preserves the relative order of elements that have equal keys.

**Figure 7.1** — Demonstrating sort stability

In [ ]:
# Each tuple: (name, grade)
# Notice Elif and Zeynep both have grade 85
students = [
    ("Elif", 85),      # appears first in original
    ("Ahmet", 92),
    ("Zeynep", 85),    # appears after Elif in original
    ("Can", 78),
]

# Stable sort: Elif stays before Zeynep (both grade 85)
stable_sorted = sorted(students, key=lambda x: x[1])
print("Stable sort (Python's sorted):")
for name, grade in stable_sorted:
    print(f"  {name:>8}: {grade}")

print("\nNotice: Elif (85) still comes before Zeynep (85) - original order preserved!")
print("This is because Python's sort is STABLE.")

---
## Part 8: Benchmarking — Insertion Sort vs Python's sorted()

> **🔮 Predict first, then measure. Does reality match your prediction?** Before running: at n=10,000, how much slower do you expect insertion sort to be compared to Python's sorted()? What about on nearly-sorted data?

**Figure 8.1** — Timing both sorting approaches on increasing list sizes

In [ ]:
import time
import random

def benchmark_sort(sort_func, data):
    """Time a sorting function."""
    data_copy = data.copy()
    start = time.perf_counter()
    sort_func(data_copy)
    return time.perf_counter() - start

def python_sort(lst):
    """Wrapper for built-in sort."""
    lst.sort()
    return lst

sizes = [100, 500, 1000, 2000, 3000, 5000, 7000, 10000]
insertion_times = []
builtin_times = []
merge_times = []

print(f"{'Size':>8}  {'Insertion (sec)':>16}  {'Merge (sec)':>14}  {'sorted() (sec)':>16}")
print("-" * 62)

for size in sizes:
    data = [random.randint(0, 100000) for _ in range(size)]
    
    t_ins = benchmark_sort(insertion_sort, data)
    t_merge = benchmark_sort(merge_sort, data)
    t_builtin = benchmark_sort(python_sort, data)
    
    insertion_times.append(t_ins)
    merge_times.append(t_merge)
    builtin_times.append(t_builtin)
    
    print(f"{size:>8,}  {t_ins:>16.6f}  {t_merge:>14.6f}  {t_builtin:>16.8f}")

**Figure 8.2** — Plotting the benchmark results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: All three on same plot
ax1.plot(sizes, insertion_times, 'ro-', label='Insertion Sort', linewidth=2, markersize=6)
ax1.plot(sizes, merge_times, 'bs-', label='Merge Sort', linewidth=2, markersize=6)
ax1.plot(sizes, builtin_times, 'g^-', label='Python sorted()', linewidth=2, markersize=6)
ax1.set_xlabel('List Size', fontsize=12)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('Sorting Algorithm Performance', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 2: Log scale to see all curves
ax2.loglog(sizes, insertion_times, 'ro-', label='Insertion Sort', linewidth=2, markersize=6)
ax2.loglog(sizes, merge_times, 'bs-', label='Merge Sort', linewidth=2, markersize=6)
ax2.loglog(sizes, builtin_times, 'g^-', label='Python sorted()', linewidth=2, markersize=6)
ax2.set_xlabel('List Size (log scale)', fontsize=12)
ax2.set_ylabel('Time (seconds, log scale)', fontsize=12)
ax2.set_title('Log-Log Scale View', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Observations:")
print("- Insertion sort grows quadratically (O(n\u00b2)) - the curve bends upward")
print("- Merge sort grows as O(n log n) - much more gradual")
print("- Python's sorted() is fastest because Timsort is highly optimized in C")

**Figure 8.3** — Insertion sort advantage: nearly sorted data

In [ ]:
import random
import time

n = 5000

# Create nearly sorted data (swap a few elements)
nearly_sorted = list(range(n))
for _ in range(10):  # Only 10 swaps
    i, j = random.randint(0, n-1), random.randint(0, n-1)
    nearly_sorted[i], nearly_sorted[j] = nearly_sorted[j], nearly_sorted[i]

# Random data
random_data = [random.randint(0, 100000) for _ in range(n)]

print(f"List size: {n}")
print(f"{'':20} {'Nearly Sorted':>14}  {'Random':>14}")
print("-" * 52)

for name, func in [("Insertion Sort", insertion_sort), ("Merge Sort", merge_sort)]:
    t_nearly = benchmark_sort(func, nearly_sorted)
    t_random = benchmark_sort(func, random_data)
    print(f"{name:20} {t_nearly:>14.6f}  {t_random:>14.6f}")

print("\nNotice: Insertion sort is FAST on nearly-sorted data (close to O(n))!")
print("This is why Timsort uses insertion sort for small/sorted runs.")

---
## Part 9: Common Errors with Sorting

**Figure 9.1** — Error: Forgetting that .sort() returns None

In [ ]:
# COMMON MISTAKE
data = [5, 3, 1, 4, 2]

# WRONG: .sort() returns None!
result = data.sort()
print(f"result = data.sort() -> {result}")
print(f"data is now sorted:    {data}")

print()

# CORRECT: Use sorted() if you want a return value
data2 = [5, 3, 1, 4, 2]
result2 = sorted(data2)
print(f"result = sorted(data) -> {result2}")
print(f"data2 is unchanged:     {data2}")

**Figure 9.2** — Error: Sorting mixed types

In [ ]:
# Python 3 cannot compare strings and integers
mixed = [3, "hello", 1, "world"]

try:
    sorted(mixed)
except TypeError as e:
    print(f"TypeError: {e}")
    print("\nFix: Make sure all elements are the same type,")
    print("or provide a key function that converts them.")

# Fix: convert everything to strings for comparison
print(f"\nUsing key=str: {sorted(mixed, key=str)}")

**Figure 9.3** — Error: Modifying a list while sorting

In [ ]:
# CAREFUL: Don't modify the comparison during sorting
counter = 0

def bad_key(x):
    """A key function with side effects (BAD!)."""
    global counter
    counter += 1
    return x

data = [5, 3, 1, 4, 2]
counter = 0
result = sorted(data, key=bad_key)
print(f"Sorted: {result}")
print(f"Key function called {counter} times for {len(data)} elements")
print("\nLesson: Key functions should be PURE (no side effects).")
print("Python guarantees key is called exactly once per element.")

---
## 🌉 Bridge to Next Week

This week we explored **sorting** — one of the most important operations in computer science.

Key takeaways:
- **Insertion Sort** is simple and efficient for small/nearly-sorted data (O(n²))
- **Merge Sort** guarantees O(n log n) but uses extra memory
- **Quick Sort** is fast in practice but has O(n²) worst case
- Python's **Timsort** combines the best of both worlds
- Always use `sorted()` or `.sort()` in practice — they're highly optimized

**Next week**, we'll dive into **Hashing** — the technique behind Python's `dict` and `set` that gives us O(1) average-case lookups. We'll learn:
- How hash functions convert keys into array indices
- What collisions are and how to handle them
- Build a simple hash table from scratch
- Benchmark `dict` vs `list` to see the dramatic speed difference

---
## 🎢 Exercises

Complete each exercise in the code cell below its description. Make sure to **run** each cell before submitting.

> **🔮 Predict first, then measure. Does reality match your prediction?** For every exercise that involves timing or complexity analysis, make your prediction BEFORE running the code.

### Easy

**EX1 — Sort a List of Names**

Write a function `sort_names(names)` that returns names sorted alphabetically (case-insensitive).

**Expected Output:**
```
sort_names(["Charlie", "alice", "Bob"]) = ["alice", "Bob", "Charlie"]
sort_names(["Zeynep", "ahmet", "Can"]) = ["ahmet", "Can", "Zeynep"]
```

<details><summary>💡 Hint</summary>
Use `sorted()` with `key=str.lower` to sort case-insensitively.
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 — Manual Insertion Sort**

Implement `my_insertion_sort(lst)` that returns a new sorted list using insertion sort (don't use the built-in sort).

**Expected Output:**
```
my_insertion_sort([64, 25, 12, 22, 11]) = [11, 12, 22, 25, 64]
my_insertion_sort([5, 1, 4, 2, 8]) = [1, 2, 4, 5, 8]
```

<details><summary>💡 Hint</summary>
Follow the pattern from Part 2: for each element starting from index 1, shift larger elements right and insert in the correct position.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 — Sort by Second Element**

Write a function `sort_by_second(pairs)` that sorts a list of tuples by their second element.

**Expected Output:**
```
sort_by_second([("a", 3), ("b", 1), ("c", 2)]) = [("b", 1), ("c", 2), ("a", 3)]
sort_by_second([("x", 10), ("y", 5), ("z", 8)]) = [("y", 5), ("z", 8), ("x", 10)]
```

<details><summary>💡 Hint</summary>
Use `sorted()` with `key=lambda x: x[1]`.
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 — Check if Sorted**

Write a function `is_sorted(lst)` that returns `True` if the list is sorted in non-decreasing order.

**Expected Output:**
```
is_sorted([1, 2, 3, 4, 5]) = True
is_sorted([1, 3, 2, 4, 5]) = False
is_sorted([5, 4, 3, 2, 1]) = False
is_sorted([]) = True
is_sorted([1, 1, 1]) = True
```

<details><summary>💡 Hint</summary>
Compare each element with the next one. If any element is greater than the next, return False.
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium

**EX5 — Count Comparisons**

Modify insertion sort to return both the sorted list and the number of comparisons made. Test with a sorted list, reversed list, and random list of size 20.

**Expected Output (format):**
```
Sorted input:   19 comparisons
Reversed input: 190 comparisons
Random input:   ~95 comparisons (varies)
```

<details><summary>💡 Hint</summary>
Add a counter that increments each time you compare two elements in the while loop.
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 — Merge Two Sorted Lists**

Write a function `merge_sorted(lst1, lst2)` that merges two already-sorted lists into one sorted list, without using `sorted()`.

**Expected Output:**
```
merge_sorted([1, 3, 5], [2, 4, 6]) = [1, 2, 3, 4, 5, 6]
merge_sorted([1, 2, 3], [4, 5, 6]) = [1, 2, 3, 4, 5, 6]
merge_sorted([], [1, 2, 3]) = [1, 2, 3]
```

<details><summary>💡 Hint</summary>
Use two pointers (i and j). Compare elements at both pointers, append the smaller one, and advance that pointer.
</details>

In [ ]:
# ✏️ [EX6] Your code here


**EX7 — Sort by Multiple Criteria**

Given a list of student tuples `(name, grade, age)`, sort by grade (descending), then by age (ascending) for ties.

**Expected Output:**
```
Input: [("Elif", 85, 21), ("Ahmet", 92, 20), ("Zeynep", 85, 19), ("Can", 92, 22)]
Output: [("Ahmet", 92, 20), ("Can", 92, 22), ("Zeynep", 85, 19), ("Elif", 85, 21)]
```

<details><summary>💡 Hint</summary>
Use `sorted()` with `key=lambda s: (-s[1], s[2])`. Negating the grade makes higher grades come first.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 — Selection Sort**

Implement **selection sort**: repeatedly find the minimum element from the unsorted portion and put it at the beginning.

**Expected Output:**
```
selection_sort([64, 25, 12, 22, 11]) = [11, 12, 22, 25, 64]
```

<details><summary>💡 Hint</summary>
For each position i, find the minimum in lst[i:], then swap it with lst[i].
</details>

In [ ]:
# ✏️ [EX8] Your code here


**EX9 — Bubble Sort**

Implement **bubble sort**: repeatedly swap adjacent elements if they're in the wrong order. Optimize by stopping early if no swaps occur in a pass.

**Expected Output:**
```
bubble_sort([64, 34, 25, 12, 22, 11, 90]) = [11, 12, 22, 25, 34, 64, 90]
```

<details><summary>💡 Hint</summary>
Use a boolean `swapped` flag. If no swaps happen in a full pass, the list is sorted and you can stop early.
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 — Sort Benchmark**

Create a benchmark that times your `insertion_sort` and Python's `sorted()` for sizes `[100, 500, 1000, 2000, 5000]` on random data. Print a table showing both times.

**Expected Output (format):**
```
Size     Insertion      sorted()
100      0.000xxx       0.000xxx
500      0.00xxxx       0.000xxx
...
```

<details><summary>💡 Hint</summary>
Use `time.perf_counter()` to measure time. Generate random data with `random.sample(range(size*10), size)`. Remember to copy the data before sorting!
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge

**EX11 — Implement Merge Sort**

Implement merge sort from scratch with both `my_merge_sort(lst)` and a helper `my_merge(left, right)` function.

**Expected Output:**
```
my_merge_sort([38, 27, 43, 3, 9, 82, 10]) = [3, 9, 10, 27, 38, 43, 82]
my_merge_sort([5, 1]) = [1, 5]
my_merge_sort([]) = []
```

<details><summary>💡 Hint</summary>
Split the list at the midpoint. Recursively sort both halves. Merge by comparing elements from both halves and building the result list.
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 — Find the K Smallest Elements**

Write `k_smallest(lst, k)` that returns the k smallest elements in sorted order. Try to be efficient (think about which sorting approach helps).

**Expected Output:**
```
k_smallest([7, 10, 4, 3, 20, 15], 3) = [3, 4, 7]
k_smallest([7, 10, 4, 3, 20, 15], 1) = [3]
k_smallest([5, 3, 1], 3) = [1, 3, 5]
```

<details><summary>💡 Hint</summary>
Simple approach: sort and take first k elements. More efficient: use partial insertion sort that stops after finding k elements, or use Python's `heapq.nsmallest()`.
</details>

In [ ]:
# ✏️ [EX12] Your code here


---
## 📮 Submission

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 1: Fill in your info below, then run this cell
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STUDENT_ID    = ""     # e.g. "2024001234"
STUDENT_NAME  = ""     # e.g. "Ahmet Y\u0131lmaz"
STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"
CLASS_CODE    = ""     # code given in class
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Don't change anything below this line
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import re as _re
_errors = []
if not _re.match(r"^\d{6,12}$", STUDENT_ID):
    _errors.append("\u274c Student ID must be 6-12 digits")
if len(STUDENT_NAME.strip().split()) < 2:
    _errors.append("\u274c Enter first and last name")
if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:
    _errors.append("\u274c Use your @istun.edu.tr email")
if len(CLASS_CODE.strip()) < 4:
    _errors.append("\u274c Invalid class code")
if _errors:
    for _e in _errors:
        print(_e)
    print("\n\u26a0\ufe0f  Fix the errors above and run this cell again.")
else:
    print(f"\u2705 Info OK \u2014 {STUDENT_NAME} ({STUDENT_ID})")
    print(f"   {STUDENT_EMAIL}")
    print(f"\n\ud83d\udc49 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 2: Run this cell to submit
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, re, os, urllib.request
WEEK = "Week_06"
URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"
try:
    _sid = STUDENT_ID.strip()
    _sname = STUDENT_NAME.strip()
    _semail = STUDENT_EMAIL.strip().lower()
    _scode = CLASS_CODE.strip().upper()
except NameError:
    raise SystemExit("\u274c Run the cell above first to set your info!")
if not _sid or not _sname or not _semail or not _scode:
    raise SystemExit("\u274c Run the cell above first \u2014 some fields are empty.")
_answers = {}
try:
    _ipy = get_ipython()
    _hist = _ipy.history_manager.get_range(output=False)
    for _sess, _line, _src in _hist:
        _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
        if _m:
            _ex_id = "ex" + _m.group(1)
            _lines = _src.split("\n")
            _clean = "\n".join(_lines[1:]).strip()
            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
except Exception:
    pass
if not _answers:
    try:
        for _src in In:
            if not _src: continue
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
    except NameError:
        pass
if not _answers:
    _nb_path = None
    try:
        _nb_path = __vsc_ipynb_file__
    except NameError:
        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]
        if len(_candidates) == 1: _nb_path = _candidates[0]
    if _nb_path and os.path.exists(str(_nb_path)):
        with open(str(_nb_path), "r", encoding="utf-8") as _f:
            _nb = json.load(_f)
        for _cell in _nb["cells"]:
            if _cell["cell_type"] != "code": continue
            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
print(f"\ud83d\udcdd Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")
if not _answers:
    print("\n\u26a0\ufe0f  No exercise answers found!")
    print("Make sure you RAN all exercise cells before submitting.")
    raise SystemExit()
_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "dsa-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")
print("\ud83d\udce1 Submitting...")
try:
    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
    _resp = urllib.request.urlopen(_req, timeout=30)
    _result = json.loads(_resp.read().decode())
    if _result.get("success"):
        print(f"\n\u2705 {_result['message']}")
        print("\ud83d\udce7 Check your email for confirmation.")
    else:
        print(f"\n\u274c {_result.get('message', 'Submission failed')}")
except Exception as _e:
    try:
        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
        urllib.request.urlopen(_req, timeout=10)
    except: pass
    print(f"\n\u26a0\ufe0f  Request sent \u2014 check your email for confirmation.")
    print(f"(If no email arrives, try again or contact your instructor)")